# Observational Confounding, Propensity Scores, and IPW Lab


## Purpose

This lab demonstrates confounding, propensity scores, and inverse probability weighting.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 5000

data = pd.DataFrame({
    "unit_id": [f"unit_{i:05d}" for i in range(1, n + 1)],
    "prior_activity": rng.normal(0, 1, size=n),
    "domain_expertise": rng.normal(0, 1, size=n),
})

data["true_tau"] = (
    0.08
    + 0.04 * (data["prior_activity"] > 0)
    + 0.03 * (data["domain_expertise"] > 0)
)

data["y0"] = (
    0.30
    + 0.08 * data["prior_activity"]
    + 0.04 * data["domain_expertise"]
    + rng.normal(0, 0.05, size=n)
)

data["y1"] = data["y0"] + data["true_tau"]

data.head()

In [ ]:
propensity = 1 / (1 + np.exp(-(-0.2 + 1.2 * data["prior_activity"])))
data["propensity"] = propensity
data["observed_treatment"] = rng.binomial(1, propensity)

data["observed_outcome"] = np.where(
    data["observed_treatment"] == 1,
    data["y1"],
    data["y0"],
)

weights = (
    data["observed_treatment"] / data["propensity"]
    + (1 - data["observed_treatment"]) / (1 - data["propensity"])
)

treated_mean = np.average(
    data.loc[data["observed_treatment"] == 1, "observed_outcome"],
    weights=weights[data["observed_treatment"] == 1],
)

control_mean = np.average(
    data.loc[data["observed_treatment"] == 0, "observed_outcome"],
    weights=weights[data["observed_treatment"] == 0],
)

ipw_estimate = treated_mean - control_mean

naive_estimate = (
    data.loc[data["observed_treatment"] == 1, "observed_outcome"].mean()
    - data.loc[data["observed_treatment"] == 0, "observed_outcome"].mean()
)

pd.DataFrame([
    {"estimate": "true_ate", "value": data["true_tau"].mean()},
    {"estimate": "naive_observational", "value": naive_estimate},
    {"estimate": "ipw_estimate", "value": ipw_estimate},
])

## Interpretation

IPW can reduce measured confounding when the propensity model is well-specified and identification assumptions are plausible.